# Cloud HAND-FIM: Python SDK submission and monitoring

This notebook imports the **generated OpenAPI Python SDK** directly to call a Cloud Run API. No CLI subprocesses are used. HydroShare authentication and a workspace are required. Jobs incur cloud costs. Run cells in order; never rerun submission to recover from a monitoring failure. Use `RESUME_JOB_ID` instead.

Browser approval delivers a session token directly to the SDK without copy-and-paste. Do not print token variables, client objects, or authentication responses. See the [deployment instructions](../readme.md). Select a Python 3.10+ kernel.

In [ ]:
%pip install httpx

from pathlib import Path
import importlib
import site
import sysconfig

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/fimbox_sdk').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the FIMBox checkout, or set ROOT to its absolute path.')
%pip install -e "$ROOT"

# Load editable-install .pth files created after this kernel started.
for package_dir in {sysconfig.get_path('purelib'), sysconfig.get_path('platlib')}:
    site.addsitedir(package_dir)
importlib.invalidate_caches()

from fimbox_sdk import Client, AuthenticatedClient
from fimbox_sdk.api.authentication import authentication_config_auth_config_get, current_session_auth_session_get
from fimbox_sdk.api.default import submit_hand_fim_jobs_hand_fim_post, get_job_status_jobs_job_id_get, get_job_logs_jobs_job_id_logs_get, create_workspace_workspaces_post, load_workspace_workspaces_open_post
from fimbox_sdk.models import AuthenticationConfig, HandFimJobRequest, JobLogs, JobStatus, JobSubmitted, Reach, Scenario, StoredHandFimResult, Workspace, WorkspaceCreate, WorkspaceOpen
from fimbox_sdk.types import UNSET


## Configure the workspace
Get the HTTPS endpoint with `terraform -chdir=terraform output -raw api_url` in a terminal. Set `WORKSPACE_LOCATION` to a HydroShare resource subprefix. Keep `CREATE_WORKSPACE=False` to open an existing workspace. For a new workspace, choose an empty subprefix under `<resource_id>/data/contents/`, set a name, and enable creation once; switch creation off after success. No bucket or permissions are provisioned. Leave `RESUME_JOB_ID` empty only for a new job. `MAX_PROCS=1` limits scenario concurrency; Cloud Run scratch files still consume worker memory. Output paths are assigned by the workspace.

In [ ]:
import os
from urllib.parse import urlsplit
from fimbox_sdk.models.hand_fim_job_request_execution_mode import (
    HandFimJobRequestExecutionMode,
)


API_URL = os.environ.get('FIMBOX_API_URL', 'http://localhost:8000').rstrip('/')
RESOURCE_ID = '7ec53d8d57ca48dc81f6ebb689844f2d'
WORKSPACE_LOCATION = f's3://resource/{RESOURCE_ID}/data/contents/workspaces/study-a'
WORKSPACE_NAME = 'Flood study'
CREATE_WORKSPACE = False
EXECUTION_MODE = HandFimJobRequestExecutionMode.STREAMING_2026
MAX_PROCS = 1
RESUME_JOB_ID = ''
POLL_SECONDS = 10
WAIT_SECONDS = 1800

origin = urlsplit(API_URL)
if not WORKSPACE_LOCATION.startswith('s3://'):
    raise ValueError('Set WORKSPACE_LOCATION to an S3 workspace location.')


## Authenticate
Run the next cell, click its login link, confirm you initiated this login, and authorize with HydroShare. The cell polls for up to ten minutes and receives the FIMBox session token automatically, including with remote kernels. An existing FIMBOX_SESSION_TOKEN environment value takes precedence. Tokens remain in memory. If login expires, is denied, or loses its connection, rerun this cell to start a new transaction.

In [ ]:
from fimbox_login import begin_login
import html
import httpx
from IPython.display import HTML, display

def call_api(operation, *, client, expected_status=200, expected_type=None, **kwargs):
    try:
        response = operation.sync_detailed(client=client, **kwargs)
    except httpx.RequestError:
        raise RuntimeError('API connection failed. If submitting, inspect your jobs before attempting another submission.') from None
    except (ValueError, KeyError, TypeError, AttributeError):
        raise RuntimeError('API response does not match the generated SDK contract.') from None
    if response.status_code != expected_status:
        detail = response.content.decode(errors='replace').strip()
        message = f'API request failed (HTTP {response.status_code})'
        raise RuntimeError(f'{message}: {detail[:1000]}' if detail else f'{message}. Check endpoint, session expiry, bucket access, and server logs.')
    if expected_type is not None and not isinstance(response.parsed, expected_type):
        raise RuntimeError('API returned an unexpected response model.')
    return response.parsed

session_token = ''
with Client(base_url=API_URL, timeout=httpx.Timeout(30), follow_redirects=False) as public_client:
    auth_config = call_api(authentication_config_auth_config_get, client=public_client, expected_type=AuthenticationConfig)
    if auth_config.authentication_enabled is not True:
        raise RuntimeError('Expected a reachable, authentication-enabled cloud API.')
    if auth_config.workspace_required is not True:
        raise RuntimeError('This API does not enforce workspaces. Upgrade the API before using this notebook.')
    session_token = os.environ.get('FIMBOX_SESSION_TOKEN', '').strip()
    if not session_token:
        browser_login = begin_login(public_client)
        display(HTML(f'<a href="{html.escape(browser_login.authorization_url, quote=True)}" target="_blank" rel="noopener noreferrer">Authorize FIMBox</a>'))
        print('Open the link and authorize. Waiting for browser approval...')
        session_token = browser_login.wait()
        del browser_login
print('Session received. Run the Connect cell below.')


### Connect using the browser-approved session

Run this cell after authentication completes. It validates the received session, then creates or opens the configured workspace. Opening is read-only and never creates missing workspaces. Every subsequent job/map request carries the workspace header. Never print the token or client object. Clear outputs before sharing.

In [ ]:
if not session_token.strip():
    raise ValueError('Run the authentication cell and authorize in the browser first.')

previous_client = globals().get('api_client')
if isinstance(previous_client, AuthenticatedClient):
    previous_client.get_httpx_client().close()
del previous_client
api_client = AuthenticatedClient(base_url=API_URL, token=session_token.strip(),
                                 timeout=httpx.Timeout(30), follow_redirects=False)
_ = call_api(current_session_auth_session_get, client=api_client)
if CREATE_WORKSPACE:
    active_workspace = call_api(create_workspace_workspaces_post, client=api_client,
                                expected_status=201, expected_type=Workspace,
                                body=WorkspaceCreate(location=WORKSPACE_LOCATION, name=WORKSPACE_NAME))
else:
    active_workspace = call_api(load_workspace_workspaces_open_post, client=api_client,
                                expected_type=Workspace, body=WorkspaceOpen(location=WORKSPACE_LOCATION))
api_client = api_client.with_headers({'X-FIMBox-Workspace': WORKSPACE_LOCATION})
job_id = RESUME_JOB_ID
print(f'Authenticated; workspace selected: {active_workspace.name}')


In [ ]:
scenarios = [Scenario(scenario_id='single-reach-scenario', reaches=[
    Reach(huc='03020201', reach_id='8780625', streamflow=7)
])]


### Load scenarios from a test JSON file

Alternatively, run this cell to replace `scenarios` with the sample file's contents. Change `scenario_path` to another scenario JSON array in `test-data/`. `Scenario.from_dict` constructs the generated SDK models, including nested reaches. This cell only reads inputs; the following submission cell starts the job.

In [ ]:
import json

scenario_path = ROOT / 'test-data' / 'single_reach.json'
scenario_data = json.loads(scenario_path.read_text(encoding='utf-8'))
scenarios = [Scenario.from_dict(item) for item in scenario_data]
print(f'Loaded {len(scenarios)} scenario(s) from {scenario_path.name}')


In [ ]:
from uuid import UUID

if RESUME_JOB_ID:
    job_id = str(UUID(RESUME_JOB_ID))
else:
    submitted = call_api(submit_hand_fim_jobs_hand_fim_post, client=api_client,
                         expected_status=202, expected_type=JobSubmitted,
                         body=HandFimJobRequest(scenarios=scenarios, max_procs=MAX_PROCS,
                                                output_prefix=UNSET, overwrite=UNSET,
                                                execution_mode=EXECUTION_MODE))
    job_id = submitted.job_id


In [ ]:
import time
from uuid import UUID

deadline = time.monotonic() + WAIT_SECONDS
log_offset = 0
while True:
    log_response = call_api(get_job_logs_jobs_job_id_logs_get, client=api_client,
                            expected_type=JobLogs, job_id=UUID(job_id), offset=log_offset)
    log_offset = log_response.next_offset
    state = log_response.status
    print(f'{time.strftime("%H:%M:%S")}  {state}', flush=True)
    if log_response.logs:
        print(log_response.logs, end='', flush=True)
    if state == 'success':
        status_response = call_api(get_job_status_jobs_job_id_get, client=api_client,
                                   expected_type=JobStatus, job_id=UUID(job_id))
        final_result = status_response.result
        if not isinstance(final_result, StoredHandFimResult):
            raise RuntimeError('Successful job has no typed output result.')
        break
    if state in {'failure', 'revoked'}:
        raise RuntimeError(f'Job ended with {state}; do not automatically resubmit.')
    remaining = deadline - time.monotonic()
    if remaining <= 0:
        raise TimeoutError('Stopped waiting. The cloud job continues; rerun this monitoring cell, not submission.')
    time.sleep(min(POLL_SECONDS, remaining))

for artifact in final_result.files:
    display_path = artifact.path.removeprefix(f'{job_id}-')
    print(display_path, artifact.size_bytes, 'bytes')


## Map retained rasters and streaming scenarios

Load retained outputs after Redis metadata expires (24 hours). The map cell derives `jobs/<job_id>/outputs` inside the active workspace, using `RESUME_JOB_ID` or the newly submitted job ID. Set `RESUME_JOB_ID`, match `EXECUTION_MODE` and scenario names to the saved run, then run setup, configuration, authentication/Connect, scenario inputs, and the map cell; skip submission and monitoring. No saved result is needed. You can instead edit `SCENARIO_PATHS` to explicit manifest or COG/TIFF paths inside the active workspace. Pan/zoom requests visible 256-pixel tiles. Blue = inundation; dry land/nodata = transparent. The helper is reloaded for existing kernels. COGs use range reads; ordinary TIFFs may require more I/O.

The workspace header and current HydroShare S3 permissions authorize each request. Paths outside the active workspace are rejected. S3 credentials stay in the API; the SDK session stays in the kernel. No credentials or signed URLs are sent to the map browser. No public S3 access or S3 browser CORS configuration is needed. OpenStreetMap provides the basemap.

Install the optional `fimbox[map]` widget dependency and enable ipyleaflet/Jupyter widgets in your frontend if the map does not render. Clear widget outputs before saving/sharing: they contain raster imagery, not credentials. If a session expires, rerun authentication/Connect and this map cell, not submission. Viewing requires a valid HydroShare session with read access and retained S3 objects, but not unexpired job metadata.

In [ ]:
import importlib
import fimbox_map
from uuid import UUID

output_map = importlib.reload(fimbox_map).output_map

if EXECUTION_MODE in (HandFimJobRequestExecutionMode.FIMSERV_2024, HandFimJobRequestExecutionMode.FIMSERV_2026):
    map_suffix = '.cog'
elif EXECUTION_MODE in (HandFimJobRequestExecutionMode.STREAMING_2024, HandFimJobRequestExecutionMode.STREAMING_2026):
    map_suffix = '/manifest.json'
else:
    raise ValueError(f'Unsupported map execution mode: {EXECUTION_MODE!r}')
map_job_id = str(UUID(RESUME_JOB_ID or job_id))
map_prefix = f's3://{active_workspace.bucket}/{active_workspace.prefix}/jobs/{map_job_id}/outputs'
SCENARIO_PATHS = [
    f'{map_prefix}/{scenario.scenario_id}{map_suffix}'
    for scenario in scenarios
]
result_map = output_map(api_client, SCENARIO_PATHS)
display(result_map)


## Compare a legacy run with streaming

Rerun a retained legacy run's scenarios with streaming and diff each scenario cell by cell. `LEGACY_PREFIX` must be a legacy run's output prefix inside the active workspace, containing `scenarios.json` and `<ScenarioID>.cog`; it defaults to `RESUME_JOB_ID` or the last submitted job. Streaming tiles and parity reports are written under the new parity job's server-assigned output prefix. A whole-pixel grid offset of up to 2 pixels is realigned before counting and reported as a grid shift. Differences are reported per scenario and do not fail the job. The server must have streaming configured. If waiting times out, do not resubmit; check `GET /jobs/{parity_job_id}`.

In [ ]:
import time
from uuid import UUID
from fimbox_sdk.api.default import submit_parity_jobs_parity_post
from fimbox_sdk.models import ParityJobRequest, ParityJobRequestExecutionMode
from fimbox_sdk.types import Unset

legacy_job_id = str(UUID(RESUME_JOB_ID or job_id))
LEGACY_PREFIX = f'{active_workspace.prefix}/jobs/{legacy_job_id}/outputs'

parity_submitted = call_api(submit_parity_jobs_parity_post, client=api_client,
                            expected_status=202, expected_type=JobSubmitted,
                            body=ParityJobRequest(legacy_bucket=active_workspace.bucket, legacy_prefix=LEGACY_PREFIX,
                                                  output_prefix=UNSET, overwrite=UNSET,
                                                  execution_mode=ParityJobRequestExecutionMode.STREAMING_2026,
                                                  ))
parity_job_id = parity_submitted.job_id

deadline = time.monotonic() + WAIT_SECONDS
log_offset = 0
while True:
    log_response = call_api(get_job_logs_jobs_job_id_logs_get, client=api_client,
                            expected_type=JobLogs, job_id=UUID(parity_job_id), offset=log_offset)
    log_offset = log_response.next_offset
    if log_response.logs:
        print(log_response.logs, end='', flush=True)
    if log_response.status == 'success':
        break
    if log_response.status in {'failure', 'revoked'}:
        raise RuntimeError(f'Parity job ended with {log_response.status}; inspect the logs above.')
    remaining = deadline - time.monotonic()
    if remaining <= 0:
        raise TimeoutError('Stopped waiting. The parity job continues; do not resubmit.')
    time.sleep(min(POLL_SECONDS, remaining))

parity_result = call_api(get_job_status_jobs_job_id_get, client=api_client,
                         expected_type=JobStatus, job_id=UUID(parity_job_id)).result
if not isinstance(parity_result, StoredHandFimResult) or isinstance(parity_result.parity, Unset):
    raise RuntimeError('Successful parity job has no comparison result.')
for comparison in parity_result.parity:
    missing = [] if isinstance(comparison.missing_reaches, Unset) else comparison.missing_reaches
    print(f"{'MATCH' if comparison.matches else 'MISMATCH'}  {comparison.scenario_id}: "
          f'{comparison.legacy_flooded_cells} legacy / {comparison.streaming_flooded_cells} streaming flooded cells, '
          f'{comparison.legacy_only_cells} legacy-only, {comparison.streaming_only_cells} streaming-only, '
          f'{len(missing)} missing reaches')
    shift = (comparison.grid_shift_rows, comparison.grid_shift_columns)
    if not any(isinstance(value, Unset) for value in shift) and any(shift):
        print(f'    realigned a grid shift of {shift[0]} rows, {shift[1]} columns '
              f'({comparison.unaligned_different_cells} cells differed before alignment)')
    if not comparison.matches:
        print(f'    diff raster: s3://{OUTPUT_BUCKET}/{comparison.diff_key}')
mismatched = [comparison.scenario_id for comparison in parity_result.parity if not comparison.matches]
print('All scenarios match.' if not mismatched else f'Streaming differs from legacy for: {", ".join(mismatched)}')